# Association Rules - Starter Notebook
Mines frequent itemsets and generates association rules (support, confidence, lift) from transaction data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations
from collections import defaultdict

In [ ]:
# Transactions: market-basket style
transactions = [
    ['milk', 'bread', 'butter'],
    ['bread', 'butter'],
    ['milk', 'bread'],
    ['milk', 'bread', 'butter', 'eggs'],
    ['bread', 'eggs'],
    ['milk', 'butter'],
    ['milk', 'eggs'],
    ['bread', 'butter', 'eggs'],
    ['milk', 'bread', 'eggs'],
    ['butter', 'eggs']
]
n_transactions = len(transactions)
print(f'Total transactions: {n_transactions}')

In [ ]:
def get_support(itemset, transactions):
    itemset = frozenset(itemset)
    return sum(1 for t in transactions if itemset.issubset(t)) / len(transactions)

def frequent_itemsets(transactions, min_support=0.3):
    items = sorted({item for t in transactions for item in t})
    freq = {}
    # 1-itemsets
    for item in items:
        sup = get_support([item], transactions)
        if sup >= min_support:
            freq[frozenset([item])] = sup
    # k-itemsets
    k = 2
    prev = list(freq.keys())
    while prev:
        candidates = [a | b for a in prev for b in prev if len(a | b) == k]
        candidates = list(set(candidates))
        new_freq = {}
        for c in candidates:
            sup = get_support(c, transactions)
            if sup >= min_support:
                new_freq[c] = sup
        freq.update(new_freq)
        prev = list(new_freq.keys())
        k += 1
    return freq

freq_sets = frequent_itemsets(transactions, min_support=0.3)
print(f'Frequent itemsets found: {len(freq_sets)}')
for fs, sup in sorted(freq_sets.items(), key=lambda x: -x[1]):
    print(f'  {set(fs)}  support={sup:.2f}')

In [ ]:
def association_rules(freq_sets, transactions, min_confidence=0.6):
    rules = []
    for itemset in freq_sets:
        if len(itemset) < 2:
            continue
        for k in range(1, len(itemset)):
            for antecedent in combinations(itemset, k):
                antecedent = frozenset(antecedent)
                consequent = itemset - antecedent
                support_full = freq_sets[itemset]
                support_ant = get_support(antecedent, transactions)
                confidence = support_full / support_ant
                support_con = get_support(consequent, transactions)
                lift = confidence / support_con
                if confidence >= min_confidence:
                    rules.append({'antecedent': set(antecedent),
                                  'consequent': set(consequent),
                                  'support': round(support_full, 3),
                                  'confidence': round(confidence, 3),
                                  'lift': round(lift, 3)})
    return pd.DataFrame(rules)

rules_df = association_rules(freq_sets, transactions, min_confidence=0.6)
print(rules_df.sort_values('lift', ascending=False).to_string(index=False))

In [ ]:
# Scatter: support vs confidence, size=lift
plt.figure(figsize=(7, 5))
sc = plt.scatter(rules_df['support'], rules_df['confidence'],
                 s=rules_df['lift'] * 80, c=rules_df['lift'],
                 cmap='viridis', alpha=0.8, edgecolors='k')
plt.colorbar(sc, label='Lift')
plt.xlabel('Support')
plt.ylabel('Confidence')
plt.title('Association Rules: Support vs Confidence (size=Lift)')
plt.tight_layout()
plt.show()